In [6]:
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision import transforms
from PIL import Image
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, npz_path):
        npz_data = np.load(npz_path)
        self.images = npz_data["images"] # (N, 3, 128, 128) in np.uint8
        self.labels = npz_data["labels"] # (N,) in np.int64
        assert self.images.shape[0] == self.labels.shape[0]
        print(f"{npz_path}: images shape {self.images.shape}, "
              f"labels shape {self.labels.shape}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx]) / 255 # convert to [0, 1] range
        label = torch.tensor(self.labels[idx])
        return image, label

train_dataset = CustomDataset("train.npz")
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)


train.npz: images shape (18900, 3, 128, 128), labels shape (18900,)


In [7]:
# sample data batch
images, labels = next(iter(train_loader))
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")

images shape: torch.Size([128, 3, 128, 128])
labels shape: torch.Size([128])


In [8]:
import matplotlib.pyplot as plt

# load label text
with open('label2type.txt', 'r') as f:
    label2type = eval(f.read())

# plot samples
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for i, ax in enumerate(axes):
    label = labels[i].item()
    # (C, H, W) to (H, W, C) for plotting
    ax.imshow(images[i].numpy().transpose(1, 2, 0))
    ax.axis('off')
    ax.set_title(f"{label}: {label2type[label]}")
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 4 Axes>

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18

# 获取标签类别数量（确保定义了 train_dataset）
num_classes = len(set(train_dataset.labels))

class ConvVAE(nn.Module):
    def __init__(self, input_channels=3, latent_channels=8):
        super().__init__()

        # --- Encoder ---

        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, stride=2, padding=1),  # -> [B, 64, 64, 64]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),             # -> [B, 128, 32, 32]
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),            # -> [B, 256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU()
        )


        # --- Bottleneck ---
        self.quant_conv = nn.Conv2d(256, latent_channels * 2, kernel_size=1)  # -> [B, 64, 16, 16]
        self.post_quant_conv = nn.Conv2d(latent_channels, 256, kernel_size=1)

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),            # -> [B,128,32,32]
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),    # -> [B,64,64,64]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 64, kernel_size=4, stride=2, padding=1),     # -> [B,64,128,128]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, input_channels, kernel_size=3, stride=1, padding=1),  # -> [B,3,128,128]
            nn.Sigmoid()
        )


        

    def preprocess(self, x):
        return 2 * x - 1  # Optional: normalize from [0,1] to [-1,1]

    def encode(self, x):
        x = self.preprocess(x)
        h = self.encoder(x)
        h = self.quant_conv(h)
        mean, logvar = torch.chunk(h, 2, dim=1)
        if self.training:
            logvar = logvar.clamp(-30.0, 20.0)
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z = mean + eps * std
        else:
            z = mean
        return z, mean, logvar

    def decode(self, z):
        h = self.post_quant_conv(z)
        return self.decoder(h)

    def forward(self, x):
        z, mean, logvar = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z, mean, logvar

In [10]:
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm


def vae_loss(x, x_recon, mean, logvar, kl_weight):
    recon_loss = F.mse_loss(x_recon, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
    final_loss = recon_loss + kl_weight * kl_loss
    return final_loss, recon_loss, kl_loss

def freeze_decoder(model):
    for param in model.decoder.parameters():
        param.requires_grad = False
    for param in model.post_quant_conv.parameters():
        param.requires_grad = False

def unfreeze_decoder(model):
    for param in model.decoder.parameters():
        param.requires_grad = True
    for param in model.post_quant_conv.parameters():
        param.requires_grad = True

def evaluate_mse(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_recon, *_ = model(x)
            loss = F.mse_loss(x_recon, x, reduction='sum')
            total_loss += loss.item()
    mse = total_loss / len(dataloader.dataset)
    return mse

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def extract_latent_and_labels(model, dataloader, device, use_mean=True):
    model.eval()
    latents, labels = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            _, z, mean, _, _ = model(x)
            latent = mean if use_mean else z
            latents.append(latent.view(latent.size(0), -1).cpu())
            labels.append(y)
    return torch.cat(latents), torch.cat(labels)

def evaluate_linear_probing(model, dataloader, device, use_mean=True):
    X, y = extract_latent_and_labels(model, dataloader, device, use_mean)
    X = X.numpy()
    y = y.numpy()
    split = int(0.8 * len(X))
    clf = LogisticRegression(max_iter=2000, solver='lbfgs', multi_class='multinomial')
    clf.fit(X[:split], y[:split])
    acc = accuracy_score(y[split:], clf.predict(X[split:]))
    print(f"Linear probing accuracy: {acc:.4f}")
    return acc




# ----- Training -----
def train_vae(model, dataloader, optimizer, device, num_epochs=200, max_kl_weight=1e-4):
    model.train()
    for epoch in range(num_epochs):
        kl_weight = max_kl_weight * epoch / num_epochs  # linear KL annealing

        loop = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{num_epochs} [KL={kl_weight:.1e}]")
        for images, _ in loop:
            x = images.to(device)

            optimizer.zero_grad()
            x_recon, z, mean, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss(x, x_recon, mean, logvar, kl_weight)
            loss.backward()
            optimizer.step()

            loop.set_postfix(
                loss=loss.item(),
                recon=recon_loss.item(),
                kl=kl_loss.item()
            )

        




# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model
model = ConvVAE(input_channels=3, latent_channels=8).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)


# Train
train_vae(model, train_loader, optimizer, device, num_epochs=1000)

Epoch 1000/1000 [KL=1.0e-04]: 100%|██████████| 148/148 [00:34<00:00,  4.34it/s, kl=1.24, loss=0.000609, recon=0.000485]


In [11]:
# ----- Visualization -----
def plot_reconstructions(model, dataloader, device, num_images=8):
    model.eval()
    with torch.no_grad():
        x = next(iter(dataloader))[0].to(device)
        x_recon, z, *_ = model(x)
        x = x.cpu().numpy()
        x_recon = x_recon.cpu().numpy()
        print(f"Latent bottleneck dimension: {z.flatten(start_dim=1).shape[1]}")

        plt.figure(figsize=(16, 4))
        for i in range(num_images):
            # Original
            plt.subplot(2, num_images, i+1)
            plt.imshow(x[i].transpose(1, 2, 0))  # (C, H, W) -> (H, W, C)
            plt.axis('off')

            # Reconstruction
            plt.subplot(2, num_images, i+1+num_images)
            plt.imshow(x_recon[i].transpose(1, 2, 0))
            plt.axis('off')

        plt.show()

plot_reconstructions(model, train_loader, device, num_images=8)

Latent bottleneck dimension: 8192


<Figure size 1600x400 with 16 Axes>

In [12]:
def evaluate_mse(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_recon, z, mean, logvar, logits = model(x)  # ✅ 完整解包
            loss = F.mse_loss(x_recon, x, reduction='sum')
            total_loss += loss.item()

    mse = total_loss / len(dataloader.dataset)
    return mse



In [13]:
probe_classifier = nn.Sequential(
    nn.Linear(8 * 32 * 32, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, num_classes)
).to(device)

In [14]:
# Metric 1: Recon MSE on test set (on value range [0, 1])
# Metric 2: Classification accuracy (linear probing with test set latents, 170 classes)
# Final Score: recon_mse / probing_accuracy (the lower the better)
torch.save(model.state_dict(), "checkpoint10.pt")